# LSTM Forward Propagation, Loss, and Backpropagation Formulae

## Complete Mathematical Derivation with Matrix Dimensions
---

## LSTM Architecture Overview

LSTM has 4 gates + cell state + hidden state:
1. **Forget Gate (f_t)**: What to forget from cell state
2. **Input Gate (i_t)**: What new information to add
3. **Cell Gate (C̃_t)**: Candidate values to add
4. **Output Gate (o_t)**: What to output from cell state
5. **Cell State (C_t)**: Long-term memory
6. **Hidden State (h_t)**: Short-term memory / output

---
## Forward Propagation

### Notation
- `F` = input dimension (features)
- `hd` = hidden dimension
- `od` = output dimension (number of classes)
- `@` = matrix multiplication
- `*` = element-wise multiplication (Hadamard product)
- `σ` = sigmoid function
- `tanh` = hyperbolic tangent

### Input at time t
```
x_t: (F × 1)      - Current input
h_{t-1}: (hd × 1) - Previous hidden state
C_{t-1}: (hd × 1) - Previous cell state
```

---
### Step 1: Forget Gate (What to forget from cell state)

```
Matrix Dimensions:
hdx1  =  hdx(hd+F)  @  (hd+F)x1  +  hdx1

concat_t = [h_{t-1}; x_t]  (concatenate vertically)
         = (hd+F) × 1
```

$$f_t = \sigma(W_f \odot \text{concat}_t + b_f)$$
$$f_t = \sigma(W_f \odot [h_{t-1}; x_t] + b_f)$$

**Expanded form:**

$$\boxed{f_t = \sigma(W_{fh} \odot h_{t-1} + W_{fx} \odot x_t + b_f)}$$

where:
- $W_f = [W_{fh} | W_{fx}]$ (concatenated horizontally)
- $W_{fh}$: $(hd \times hd)$
- $W_{fx}$: $(hd \times F)$
- $b_f$: $(hd \times 1)$
- $f_t$: $(hd \times 1)$ - values in $[0, 1]$

**Sigmoid function:**
$$\sigma(z) = \frac{1}{1 + \exp(-z)}$$
$$\sigma'(z) = \sigma(z) \times (1 - \sigma(z))$$

---
### Step 2: Input Gate (What new information to add)

```
Matrix Dimensions:
hdx1  =  hdx(hd+F)  @  (hd+F)x1  +  hdx1
```

$$i_t = \sigma(W_i \odot \text{concat}_t + b_i)$$
$$i_t = \sigma(W_i \odot [h_{t-1}; x_t] + b_i)$$

**Expanded form:**

$$\boxed{i_t = \sigma(W_{ih} \odot h_{t-1} + W_{ix} \odot x_t + b_i)}$$

where:
- $W_i = [W_{ih} | W_{ix}]$
- $W_{ih}$: $(hd \times hd)$
- $W_{ix}$: $(hd \times F)$
- $b_i$: $(hd \times 1)$
- $i_t$: $(hd \times 1)$ - values in $[0, 1]$

---
### Step 3: Cell Gate / Candidate Values (New information to consider)

```
Matrix Dimensions:
hdx1  =  hdx(hd+F)  @  (hd+F)x1  +  hdx1
```

$$\tilde{C}_t = \tanh(W_C \odot \text{concat}_t + b_C)$$
$$\tilde{C}_t = \tanh(W_C \odot [h_{t-1}; x_t] + b_C)$$

**Expanded form:**

$$\boxed{\tilde{C}_t = \tanh(W_{Ch} \odot h_{t-1} + W_{Cx} \odot x_t + b_C)}$$

where:
- $W_C = [W_{Ch} | W_{Cx}]$
- $W_{Ch}$: $(hd \times hd)$
- $W_{Cx}$: $(hd \times F)$
- $b_C$: $(hd \times 1)$
- $\tilde{C}_t$: $(hd \times 1)$ - values in $[-1, 1]$

**Tanh function:**
$$\tanh(z) = \frac{\exp(z) - \exp(-z)}{\exp(z) + \exp(-z)}$$
$$\tanh'(z) = 1 - \tanh^2(z)$$

---
### Step 4: Update Cell State (Forget old + Add new)

```
Matrix Dimensions:
hdx1  =  hdx1  *  hdx1  +  hdx1  *  hdx1
```

$$\boxed{C_t = f_t * C_{t-1} + i_t * \tilde{C}_t}$$

**Interpretation:**
- $f_t * C_{t-1}$: Keep relevant parts of old memory
- $i_t * \tilde{C}_t$: Add relevant parts of new information
- Element-wise operations (Hadamard product)

where:
- $C_t$: $(hd \times 1)$ - Updated cell state

---
### Step 5: Output Gate (What to output from cell state)

```
Matrix Dimensions:
hdx1  =  hdx(hd+F)  @  (hd+F)x1  +  hdx1
```

$$o_t = \sigma(W_o \odot \text{concat}_t + b_o)$$
$$o_t = \sigma(W_o \odot [h_{t-1}; x_t] + b_o)$$

**Expanded form:**

$$\boxed{o_t = \sigma(W_{oh} \odot h_{t-1} + W_{ox} \odot x_t + b_o)}$$

where:
- $W_o = [W_{oh} | W_{ox}]$
- $W_{oh}$: $(hd \times hd)$
- $W_{ox}$: $(hd \times F)$
- $b_o$: $(hd \times 1)$
- $o_t$: $(hd \times 1)$ - values in $[0, 1]$

---
### Step 6: Update Hidden State (Output filtered cell state)

```
Matrix Dimensions:
hdx1  =  hdx1  *  hdx1
```

$$\boxed{h_t = o_t * \tanh(C_t)}$$

**Interpretation:**
- $\tanh(C_t)$: Squash cell state to $[-1, 1]$
- $o_t * \tanh(C_t)$: Filter what to output

where:
- $h_t$: $(hd \times 1)$ - Updated hidden state

---
### Step 7: Output Layer (Classification)

```
Matrix Dimensions:
odx1  =  odxhd  @  hdx1  +  odx1
```

$$\text{logits}_t = W_y \odot h_t + b_y$$
$$y_t = \text{softmax}(\text{logits}_t)$$

**Softmax:**
$$y_t[i] = \frac{\exp(\text{logits}_t[i])}{\sum_j \exp(\text{logits}_t[j])}$$

where:
- $W_y$: $(od \times hd)$
- $b_y$: $(od \times 1)$
- $\text{logits}_t$: $(od \times 1)$
- $y_t$: $(od \times 1)$ - probability distribution

---
## Loss Function

$$L_t = -\log(y_t[c])$$

where $c$ is the correct class index

**Full form:**
$$L_t = -\sum_i \text{target}_t[i] \times \log(y_t[i])$$

Since $\text{target}_t[i] = 0$ for all $i \neq c$, this simplifies to:
$$L_t = -\log(y_t[c])$$

**Total loss over sequence:**
$$L = \frac{1}{T} \times \sum_t L_t$$

---
## Backpropagation Through Time (BPTT)

### Gradient of Loss w.r.t. Output

```
Matrix Dimensions: odx1
```

$$\frac{\partial L_t}{\partial y_t[i]} = \begin{cases} 
-\frac{1}{y_t[c]} & \text{if } i = c \text{ (correct class)} \\
0 & \text{if } i \neq c \text{ (other classes)}
\end{cases}$$

### Gradient w.r.t. Logits (Softmax + Cross-Entropy Combined)

```
Matrix Dimensions: odx1
```

$$\boxed{\frac{\partial L_t}{\partial \text{logits}_t} = y_t - \text{one\_hot}(c)}$$

$$\frac{\partial L_t}{\partial \text{logits}_t[i]} = \begin{cases} 
y_t[i] - 1 & \text{if } i = c \text{ (correct class)} \\
y_t[i] & \text{if } i \neq c \text{ (other classes)}
\end{cases}$$

---
### Gradient w.r.t. Output Layer Parameters

```
Matrix Dimensions:
∂L_t/∂W_y: (od × hd)
∂L_t/∂b_y: (od × 1)
```

$$\boxed{\frac{\partial L_t}{\partial W_y} = \frac{\partial L_t}{\partial \text{logits}_t} \odot h_t^T = (y_t - \text{one\_hot}(c)) \odot h_t^T}$$

$$\boxed{\frac{\partial L_t}{\partial b_y} = \frac{\partial L_t}{\partial \text{logits}_t} = (y_t - \text{one\_hot}(c))}$$

---
### Gradient w.r.t. Hidden State

```
Matrix Dimensions: hdx1
```

$$\boxed{\frac{\partial L_t}{\partial h_t} = W_y^T \odot \frac{\partial L_t}{\partial \text{logits}_t} + \frac{\partial L_{t+1}}{\partial h_t}}$$

$$= W_y^T \odot (y_t - \text{one\_hot}(c)) + dh_{\text{next}}$$

where $dh_{\text{next}}$ comes from backprop through time

**Note:** At the last timestep $T$, $dh_{\text{next}} = 0$

---
### Gradient w.r.t. Output Gate

**Step 1: Gradient w.r.t. $o_t$ (before sigmoid)**

$$\frac{\partial L_t}{\partial o_t} = \frac{\partial L_t}{\partial h_t} * \tanh(C_t) = dh_t * \tanh(C_t)$$

where $*$ is element-wise multiplication

**Step 2: Gradient through sigmoid activation**

$$\frac{\partial L_t}{\partial (W_o \odot \text{concat}_t + b_o)} = \frac{\partial L_t}{\partial o_t} * \sigma'(W_o \odot \text{concat}_t + b_o)$$

$$= \frac{\partial L_t}{\partial o_t} * o_t * (1 - o_t)$$

Let's call this: $do_{\text{raw}_t} = \frac{\partial L_t}{\partial o_t} * o_t * (1 - o_t)$

```
Matrix Dimensions:
do_raw_t: (hd × 1)
```

$$\boxed{\frac{\partial L_t}{\partial W_o} = do_{\text{raw}_t} \odot \text{concat}_t^T = do_{\text{raw}_t} \odot [h_{t-1}; x_t]^T}$$

$$\boxed{\frac{\partial L_t}{\partial b_o} = do_{\text{raw}_t}}$$

---
### Gradient w.r.t. Cell State

```
Matrix Dimensions: hdx1
```

$$\boxed{\frac{\partial L_t}{\partial C_t} = \frac{\partial L_t}{\partial h_t} * o_t * (1 - \tanh^2(C_t)) + \frac{\partial L_{t+1}}{\partial C_t}}$$

$$= dh_t * o_t * (1 - \tanh^2(C_t)) + dC_{\text{next}}$$

where:
- First term: gradient from current hidden state
- Second term: gradient from next timestep (BPTT)
- $dC_{\text{next}}$ comes from backprop through time

**Note:** At the last timestep $T$, $dC_{\text{next}} = 0$

---
### Gradient w.r.t. Cell Gate (Candidate Values)

**Step 1: Gradient w.r.t. $\tilde{C}_t$**

$$\frac{\partial L_t}{\partial \tilde{C}_t} = \frac{\partial L_t}{\partial C_t} * i_t = dC_t * i_t$$

**Step 2: Gradient through tanh activation**

$$\frac{\partial L_t}{\partial (W_C \odot \text{concat}_t + b_C)} = \frac{\partial L_t}{\partial \tilde{C}_t} * (1 - \tilde{C}_t^2)$$

Let's call this: $d\tilde{C}_{\text{raw}_t} = \frac{\partial L_t}{\partial \tilde{C}_t} * (1 - \tilde{C}_t^2)$

```
Matrix Dimensions:
dC_tilde_raw_t: (hd × 1)
```

$$\boxed{\frac{\partial L_t}{\partial W_C} = d\tilde{C}_{\text{raw}_t} \odot \text{concat}_t^T = d\tilde{C}_{\text{raw}_t} \odot [h_{t-1}; x_t]^T}$$

$$\boxed{\frac{\partial L_t}{\partial b_C} = d\tilde{C}_{\text{raw}_t}}$$

---
### Gradient w.r.t. Input Gate

**Step 1: Gradient w.r.t. $i_t$ (before sigmoid)**

$$\frac{\partial L_t}{\partial i_t} = \frac{\partial L_t}{\partial C_t} * \tilde{C}_t = dC_t * \tilde{C}_t$$

**Step 2: Gradient through sigmoid activation**

$$\frac{\partial L_t}{\partial (W_i \odot \text{concat}_t + b_i)} = \frac{\partial L_t}{\partial i_t} * \sigma'(W_i \odot \text{concat}_t + b_i)$$

$$= \frac{\partial L_t}{\partial i_t} * i_t * (1 - i_t)$$

Let's call this: $di_{\text{raw}_t} = \frac{\partial L_t}{\partial i_t} * i_t * (1 - i_t)$

```
Matrix Dimensions:
di_raw_t: (hd × 1)
```

$$\boxed{\frac{\partial L_t}{\partial W_i} = di_{\text{raw}_t} \odot \text{concat}_t^T = di_{\text{raw}_t} \odot [h_{t-1}; x_t]^T}$$

$$\boxed{\frac{\partial L_t}{\partial b_i} = di_{\text{raw}_t}}$$

---
### Gradient w.r.t. Forget Gate

**Step 1: Gradient w.r.t. $f_t$ (before sigmoid)**

$$\frac{\partial L_t}{\partial f_t} = \frac{\partial L_t}{\partial C_t} * C_{t-1} = dC_t * C_{t-1}$$

**Step 2: Gradient through sigmoid activation**

$$\frac{\partial L_t}{\partial (W_f \odot \text{concat}_t + b_f)} = \frac{\partial L_t}{\partial f_t} * \sigma'(W_f \odot \text{concat}_t + b_f)$$

$$= \frac{\partial L_t}{\partial f_t} * f_t * (1 - f_t)$$

Let's call this: $df_{\text{raw}_t} = \frac{\partial L_t}{\partial f_t} * f_t * (1 - f_t)$

```
Matrix Dimensions:
df_raw_t: (hd × 1)
```

$$\boxed{\frac{\partial L_t}{\partial W_f} = df_{\text{raw}_t} \odot \text{concat}_t^T = df_{\text{raw}_t} \odot [h_{t-1}; x_t]^T}$$

$$\boxed{\frac{\partial L_t}{\partial b_f} = df_{\text{raw}_t}}$$

---
### Gradient w.r.t. Previous Hidden State (for BPTT)

```
Matrix Dimensions: hdx1
```

$$\boxed{\frac{\partial L_t}{\partial h_{t-1}} = W_{fh}^T \odot df_{\text{raw}_t} + W_{ih}^T \odot di_{\text{raw}_t} + W_{Ch}^T \odot d\tilde{C}_{\text{raw}_t} + W_{oh}^T \odot do_{\text{raw}_t}}$$

This gradient flows back to the previous timestep

---
### Gradient w.r.t. Previous Cell State (for BPTT)

```
Matrix Dimensions: hdx1
```

$$\boxed{\frac{\partial L_t}{\partial C_{t-1}} = \frac{\partial L_t}{\partial C_t} * f_t}$$

This gradient flows back to the previous timestep

---
## Summary: Complete BPTT Algorithm

### Forward Pass (for all t = 1 to T)
```
1. concat_t = [h_{t-1}; x_t]
2. f_t = σ(W_f @ concat_t + b_f)
3. i_t = σ(W_i @ concat_t + b_i)
4. C̃_t = tanh(W_C @ concat_t + b_C)
5. C_t = f_t * C_{t-1} + i_t * C̃_t
6. o_t = σ(W_o @ concat_t + b_o)
7. h_t = o_t * tanh(C_t)
8. logits_t = W_y @ h_t + b_y
9. y_t = softmax(logits_t)
10. L_t = -log(y_t[c])
```

### Backward Pass (for all t = T to 1)
```
Initialize:
  dh_next = 0
  dC_next = 0

For each timestep t (from T to 1):

1. Output layer gradients:
   dlogits_t = y_t - one_hot(c)
   dW_y += dlogits_t @ h_t.T
   db_y += dlogits_t

2. Hidden state gradient:
   dh_t = W_y.T @ dlogits_t + dh_next

3. Output gate gradients:
   do_t = dh_t * tanh(C_t)
   do_raw_t = do_t * o_t * (1 - o_t)
   dW_o += do_raw_t @ concat_t.T
   db_o += do_raw_t

4. Cell state gradient:
   dC_t = dh_t * o_t * (1 - tanh²(C_t)) + dC_next

5. Cell gate gradients:
   dC_tilde_t = dC_t * i_t
   dC_tilde_raw_t = dC_tilde_t * (1 - C̃_t²)
   dW_C += dC_tilde_raw_t @ concat_t.T
   db_C += dC_tilde_raw_t

6. Input gate gradients:
   di_t = dC_t * C̃_t
   di_raw_t = di_t * i_t * (1 - i_t)
   dW_i += di_raw_t @ concat_t.T
   db_i += di_raw_t

7. Forget gate gradients:
   df_t = dC_t * C_{t-1}
   df_raw_t = df_t * f_t * (1 - f_t)
   dW_f += df_raw_t @ concat_t.T
   db_f += df_raw_t

8. Gradients for next iteration (BPTT):
   dh_next = W_fh.T @ df_raw_t + W_ih.T @ di_raw_t + 
             W_Ch.T @ dC_tilde_raw_t + W_oh.T @ do_raw_t
   dC_next = dC_t * f_t
```

---
## Summary Tables

### Forward Pass Formulas

| Step | Variable | Formula | Dimensions | Description |
|------|----------|---------|------------|-------------|
| 0 | $\text{concat}_t$ | $[h_{t-1}; x_t]$ | $(hd+F) \times 1$ | Concatenated input |
| 1 | $f_t$ | $\sigma(W_f \odot \text{concat}_t + b_f)$ | $(hd \times 1)$ | Forget gate |
| 2 | $i_t$ | $\sigma(W_i \odot \text{concat}_t + b_i)$ | $(hd \times 1)$ | Input gate |
| 3 | $\tilde{C}_t$ | $\tanh(W_C \odot \text{concat}_t + b_C)$ | $(hd \times 1)$ | Cell gate (candidate) |
| 4 | $C_t$ | $f_t * C_{t-1} + i_t * \tilde{C}_t$ | $(hd \times 1)$ | Cell state |
| 5 | $o_t$ | $\sigma(W_o \odot \text{concat}_t + b_o)$ | $(hd \times 1)$ | Output gate |
| 6 | $h_t$ | $o_t * \tanh(C_t)$ | $(hd \times 1)$ | Hidden state |
| 7 | $\text{logits}_t$ | $W_y \odot h_t + b_y$ | $(od \times 1)$ | Output logits |
| 8 | $y_t$ | $\text{softmax}(\text{logits}_t)$ | $(od \times 1)$ | Output probabilities |
| 9 | $L_t$ | $-\log(y_t[c])$ | scalar | Loss at time $t$ |

**Notation:**
- $hd$ = hidden dimension
- $F$ = input/feature dimension
- $od$ = output dimension (number of classes)
- $\odot$ = matrix multiplication
- $*$ = element-wise multiplication (Hadamard product)

**Total Parameters Used in Forward Pass:**

| Component | Parameters | Count |
|-----------|------------|-------|
| Forget Gate | $W_f, b_f$ | $hd \times (hd+F) + hd$ |
| Input Gate | $W_i, b_i$ | $hd \times (hd+F) + hd$ |
| Cell Gate | $W_C, b_C$ | $hd \times (hd+F) + hd$ |
| Output Gate | $W_o, b_o$ | $hd \times (hd+F) + hd$ |
| Output Layer | $W_y, b_y$ | $od \times hd + od$ |
| **Total** | | $4hd(hd+F+1) + od(hd+1)$ |

**Expanded:**
$$\text{Total} = 4hd^2 + 4hd \times F + 4hd + od \times hd + od$$

### Backpropagation Gradient Formulas

#### Output Layer Gradients

| Parameter | Gradient Formula | Dimensions |
|-----------|------------------|------------|
| $W_y$ | $(y_t - \text{one\_hot}(c)) \odot h_t^T$ | $(od \times hd)$ |
| $b_y$ | $(y_t - \text{one\_hot}(c))$ | $(od \times 1)$ |

#### Hidden State Gradient

| Variable | Gradient Formula | Dimensions |
|----------|------------------|------------|
| $h_t$ | $W_y^T \odot (y_t - \text{one\_hot}(c)) + dh_{\text{next}}$ | $(hd \times 1)$ |

#### Gate Gradients (Forget Gate)

| Step | Variable | Formula | Dimensions |
|------|----------|---------|------------|
| 1 | $\frac{\partial L_t}{\partial f_t}$ | $dC_t * C_{t-1}$ | $(hd \times 1)$ |
| 2 | $df_{\text{raw}_t}$ | $\frac{\partial L_t}{\partial f_t} * f_t * (1 - f_t)$ | $(hd \times 1)$ |
| 3 | $\frac{\partial L_t}{\partial W_f}$ | $df_{\text{raw}_t} \odot \text{concat}_t^T$ | $(hd \times (hd+F))$ |
| 4 | $\frac{\partial L_t}{\partial b_f}$ | $df_{\text{raw}_t}$ | $(hd \times 1)$ |

#### Gate Gradients (Input Gate)

| Step | Variable | Formula | Dimensions |
|------|----------|---------|------------|
| 1 | $\frac{\partial L_t}{\partial i_t}$ | $dC_t * \tilde{C}_t$ | $(hd \times 1)$ |
| 2 | $di_{\text{raw}_t}$ | $\frac{\partial L_t}{\partial i_t} * i_t * (1 - i_t)$ | $(hd \times 1)$ |
| 3 | $\frac{\partial L_t}{\partial W_i}$ | $di_{\text{raw}_t} \odot \text{concat}_t^T$ | $(hd \times (hd+F))$ |
| 4 | $\frac{\partial L_t}{\partial b_i}$ | $di_{\text{raw}_t}$ | $(hd \times 1)$ |

#### Gate Gradients (Cell Gate)

| Step | Variable | Formula | Dimensions |
|------|----------|---------|------------|
| 1 | $\frac{\partial L_t}{\partial \tilde{C}_t}$ | $dC_t * i_t$ | $(hd \times 1)$ |
| 2 | $d\tilde{C}_{\text{raw}_t}$ | $\frac{\partial L_t}{\partial \tilde{C}_t} * (1 - \tilde{C}_t^2)$ | $(hd \times 1)$ |
| 3 | $\frac{\partial L_t}{\partial W_C}$ | $d\tilde{C}_{\text{raw}_t} \odot \text{concat}_t^T$ | $(hd \times (hd+F))$ |
| 4 | $\frac{\partial L_t}{\partial b_C}$ | $d\tilde{C}_{\text{raw}_t}$ | $(hd \times 1)$ |

#### Gate Gradients (Output Gate)

| Step | Variable | Formula | Dimensions |
|------|----------|---------|------------|
| 1 | $\frac{\partial L_t}{\partial o_t}$ | $dh_t * \tanh(C_t)$ | $(hd \times 1)$ |
| 2 | $do_{\text{raw}_t}$ | $\frac{\partial L_t}{\partial o_t} * o_t * (1 - o_t)$ | $(hd \times 1)$ |
| 3 | $\frac{\partial L_t}{\partial W_o}$ | $do_{\text{raw}_t} \odot \text{concat}_t^T$ | $(hd \times (hd+F))$ |
| 4 | $\frac{\partial L_t}{\partial b_o}$ | $do_{\text{raw}_t}$ | $(hd \times 1)$ |

#### Cell State and Hidden State Gradients (for BPTT)

| Variable | Formula | Dimensions | Description |
|----------|---------|------------|-------------|
| $\frac{\partial L_t}{\partial C_t}$ | $dh_t * o_t * (1 - \tanh^2(C_t)) + dC_{\text{next}}$ | $(hd \times 1)$ | Cell state gradient |
| $\frac{\partial L_t}{\partial C_{t-1}}$ | $dC_t * f_t$ | $(hd \times 1)$ | Previous cell state gradient |
| $\frac{\partial L_t}{\partial h_{t-1}}$ | $W_{fh}^T \odot df_{\text{raw}_t} + W_{ih}^T \odot di_{\text{raw}_t} + W_{Ch}^T \odot d\tilde{C}_{\text{raw}_t} + W_{oh}^T \odot do_{\text{raw}_t}$ | $(hd \times 1)$ | Previous hidden state gradient |

### Complete Parameter List

| Parameter | Dimensions | Description |
|-----------|------------|-------------|
| $W_f$ | $(hd \times (hd+F))$ | Forget gate weights |
| $b_f$ | $(hd \times 1)$ | Forget gate bias |
| $W_i$ | $(hd \times (hd+F))$ | Input gate weights |
| $b_i$ | $(hd \times 1)$ | Input gate bias |
| $W_C$ | $(hd \times (hd+F))$ | Cell gate weights |
| $b_C$ | $(hd \times 1)$ | Cell gate bias |
| $W_o$ | $(hd \times (hd+F))$ | Output gate weights |
| $b_o$ | $(hd \times 1)$ | Output gate bias |
| $W_y$ | $(od \times hd)$ | Output layer weights |
| $b_y$ | $(od \times 1)$ | Output layer bias |

**Total Parameters:**
$$4 \times [hd \times (hd + F) + hd] + od \times hd + od$$
$$= 4hd^2 + 4hd \times F + 4hd + od \times hd + od$$

---
## Key Differences from Vanilla RNN

### Vanilla RNN:
$$h_t = \tanh(W_{hh} \odot h_{t-1} + W_{xh} \odot x_t + b_h)$$

Gradient flows through:
$$\frac{\partial h_t}{\partial h_{t-1}} = W_{hh}^T * (1 - h_t^2)$$

**Problem:** Repeated multiplication causes vanishing gradient

### LSTM:
$$\boxed{C_t = f_t * C_{t-1} + i_t * \tilde{C}_t}$$

Gradient flows through:
$$\boxed{\frac{\partial C_t}{\partial C_{t-1}} = f_t}$$

**Solution:** Additive path ($+$ instead of $*$) preserves gradient!

### Why LSTM solves vanishing gradient:
- Cell state uses **addition** ($C_t = f_t * C_{t-1} + ...$)
- Gradient flows through **multiplication by $f_t$** (not repeated matrix multiplication)
- When $f_t \approx 1$, gradient flows unchanged (gradient highway!)
- Gates learn when to preserve vs. update information

---
## Gradient Clipping (Important for LSTM)

After computing all gradients, clip them to prevent exploding gradients:

```python
For each gradient g in [dW_f, db_f, dW_i, db_i, dW_C, db_C, dW_o, db_o, dW_y, db_y]:
    g = clip(g, -threshold, threshold)
```

Common threshold: 5.0

---
## Parameter Count Comparison

### Vanilla RNN:
$$\text{Parameters} = hd \times hd + hd \times F + hd + od \times hd + od$$
$$= hd^2 + hd \times F + hd + od \times hd + od$$

### LSTM:
$$\text{Parameters} = 4 \times (hd \times (hd + F) + hd) + od \times hd + od$$
$$= 4 \times (hd^2 + hd \times F + hd) + od \times hd + od$$
$$\approx 4 \times \text{vanilla RNN}$$

**Why 4×?** Four gates (forget, input, cell, output) each with their own weights!

---
## Implementation Tips

1. **Concatenation vs Separate Matrices:**
   - Can use `concat_t = [h_{t-1}; x_t]` with single weight matrix
   - Or use separate `W_fh, W_fx` matrices
   - Both are equivalent, concatenation is more efficient

2. **Numerical Stability:**
   - Clip gradients to prevent explosion
   - Use stable softmax: `softmax(x - max(x))`
   - Initialize forget gate bias to 1.0 (helps learning)

3. **Initialization:**
   - Xavier/Glorot initialization for weights
   - Forget gate bias: 1.0 (default to remembering)
   - Other biases: 0.0

4. **Gradient Checking:**
   - Verify gradients numerically for small examples
   - Check each gate separately
   - Ensure dimensions match throughout

---
## Computational Complexity

### Forward Pass:
```
Time: O(T × (hd² + hd×F + od×hd))
Space: O(T × hd)  (store all hidden states for backprop)
```

### Backward Pass:
```
Time: O(T × (hd² + hd×F + od×hd))
Space: O(hd)  (only need current gradients)
```

**Note:** LSTM is ~4× slower than vanilla RNN due to 4 gates

---
## Visualization of Gradient Flow

### Vanilla RNN:
```
L → h_T → h_{T-1} → ... → h_1 → h_0
    ↓      ↓              ↓
  Gradient decays exponentially (×W_hh each step)
```

### LSTM:
```
L → C_T → C_{T-1} → ... → C_1 → C_0
    ↓      ↓              ↓
  Gradient preserved (×f_t ≈ 1 each step)
```

**The cell state $C_t$ acts as a "gradient highway"!**

---
**End of LSTM Formulae Document**